In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


In [ ]:
"""
1. To predict the sales price of houses based on a number of parameters.
2. Framed as regression problem .
3. Root Mean Squared Log Error is used to measure performance of the model.

"""

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
%%bash

ls /kaggle/input/house-prices-advanced-regression-techniques

#cat /kaggle/input/house-prices-advanced-regression-techniques/train.csv | head -n 5

In [ ]:
#pd.read_csv?

train_df = pd.read_csv( "/kaggle/input/house-prices-advanced-regression-techniques/train.csv",index_col="Id",)
test_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv",index_col="Id")

In [ ]:
train_targets = train_df.loc[:, "SalePrice"]
#test_df.head()

In [ ]:
housing_df = train_df
housing_df.info()

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
"""
-There are 33 numerical features (area, no of rooms etc) and categorical fetaures like quality of houses, geographical location.

-Few features like BsmtArea ,Pool Area have NaN values since the the data point doesn't have have a basement or pool. The pre-processing
pipeline takes care of it.

-Outliers are there for some features like MasonVeneerArea have few outliers. Outliers can be identified from scatter plots.

- Features like Floor area number of rooms, lot frontage area are very useful attributes for this task. However, there are features that dont make much
impact like categorical data which have no correlation with sale price. A scatter plot will help us detect this. If an feature has any correlation like linear or
polynomial , then features must be added to the pipeline.

- Feature selection tool from sci-kit learn is handy and helps us detect variables that effect house price the most,which in turn reduces the dimensionality
of the problem.

- From plot 2 below newer houses have slightly higher prices, but there are few outliers in the case of houses built in 1890s.
"""

In [ ]:
#Plot 1
_ = train_df.plot(
    kind="scatter",
    x="MasVnrArea",
    y="SalePrice",
    grid=True,
    alpha=0.25
)

In [ ]:
#Plot2
_ = train_df.plot(
    kind="scatter",
    x="YearBuilt",
    y="SalePrice",
    grid=True,
    alpha=0.25
)

In [ ]:
column_name = 'GarageCond'
#train_df[column_name].count()
train_df[column_name].value_counts()

In [ ]:
## Feature Selection study

from sklearn.feature_selection import SelectKBest, f_regression

X = train_df.drop('SalePrice', axis=1)
y = train_df['SalePrice']

# Keep only numerical columns
X_num = X.select_dtypes(include=['int64', 'float64'])

# Fill NaNs with the median (safer than mean for skewed data)
X_num = X_num.fillna(X_num.median())

# Apply feature selection
selector = SelectKBest(score_func=f_regression, k=36)
X_new = selector.fit_transform(X_num, y)

selected_features = X_num.columns[selector.get_support()]
print("Selected numeric features:")
print(selected_features.tolist())

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
train_df.select_dtypes(include=np.int64)

In [ ]:
FEATURES_DTYPES = {
    "LotArea": np.float64,
    "YearBuilt": np.float64,
    "YearRemodAdd": np.float64,
    "BsmtFinSF1": np.float64,
    "BsmtFinSF2": np.float64,
    "BsmtUnfSF": np.float64,
    "TotalBsmtSF": np.float64,
    "1stFlrSF": np.float64,
    "2ndFlrSF": np.float64,
    "LowQualFinSF": np.float64,
    "GrLivArea": np.float64,
    "GarageArea": np.float64,
    "WoodDeckSF": np.float64,
    "OpenPorchSF": np.float64,
    "EnclosedPorch": np.float64,
    "3SsnPorch": np.float64,
    "ScreenPorch": np.float64,
    "PoolArea": np.float64,
    "MiscVal": np.float64,
    "GarageYrBlt" : np.int64,
    "MSSubClass" : str,
    "OverallQual" : str,
    "OverallCond" : str
}


def convert_dtypes(df):
    return df.astype(
        FEATURES_DTYPES
    )

In [ ]:
from sklearn import pipeline, preprocessing
from sklearn.impute import SimpleImputer
from sklearn import compose

In [ ]:
""""train_df.dropna(
    axis="columns",
    how="any"
)
"""
def drop_features_with_missing_values(df):
    return df.dropna(
        axis="columns",
        how="any"
    )

In [ ]:
def safe_log1p(X):
    return np.log1p(np.clip(X, a_min=0, a_max=None))

def safe_expm1(x):
    x = np.clip(x, a_min=None, a_max=20)
    return np.expm1(x)

def drop_non_numeric_features(df):
    return df.select_dtypes(include=[np.float64,np.int64])

def fill_missing_vals(df):
        feat_list = ["LotArea",
                "BsmtFinSF1",
                "BsmtFinSF2",
                "BsmtUnfSF",
                "TotalBsmtSF",
                "1stFlrSF",
                "2ndFlrSF",
                "LowQualFinSF",
                "GrLivArea",
                "GarageArea",
                "WoodDeckSF",
                "OpenPorchSF",
                "EnclosedPorch",
                "3SsnPorch",
                "ScreenPorch",
                "PoolArea",
                "MiscVal","YearBuilt",
                "YearRemodAdd",
                "MasVnrArea",
                "LotFrontage",
                "GarageYrBlt",
                "YrSold",
                "MoSold",
                "GarageCars",
                "BsmtFullBath",
                "BsmtHalfBath",
                "FullBath",
                "HalfBath",
                "BedroomAbvGr",
                "KitchenAbvGr",
                "TotRmsAbvGrd",
                "Fireplaces"]
        for i in feat_list:
            df.fillna({i:0}, inplace=True)
        
        return df



In [ ]:
features_preparation_pipeline1 = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=fill_missing_vals,
    ),
    preprocessing.FunctionTransformer(
        func=convert_dtypes,
    ),
    preprocessing.FunctionTransformer(
        func=drop_non_numeric_features,
    ),

    compose.make_column_transformer(
        (
            pipeline.make_pipeline(
                preprocessing.FunctionTransformer(
                    func=safe_log1p,
                    inverse_func=safe_expm1
                ),
                preprocessing.RobustScaler(
                    unit_variance=True
                ),
                verbose=True,
            ),
            [
                "LotArea",
                "BsmtFinSF1",
                "BsmtFinSF2",
                "BsmtUnfSF",
                "TotalBsmtSF",
                "1stFlrSF",
                "2ndFlrSF",
                "LowQualFinSF",
                "GrLivArea",
                "GarageArea",
                "WoodDeckSF",
                "OpenPorchSF",
                "EnclosedPorch",
                "3SsnPorch",
                "ScreenPorch",
                "PoolArea",
                "MiscVal"
            ]
        ),
        (
            pipeline.make_pipeline(
                preprocessing.RobustScaler(
                    unit_variance=True
                ),
                verbose=True,
            ),
            [
                "YearBuilt",
                "YearRemodAdd",
                "MasVnrArea",
                "LotFrontage",
                "GarageYrBlt",
                "YrSold",
                "MoSold",
                "GarageCars",
                "BsmtFullBath",
                "BsmtHalfBath",
                "FullBath",
                "HalfBath",
                "BedroomAbvGr",
                "KitchenAbvGr",
                "TotRmsAbvGrd",
                "Fireplaces"        
            ]
        ),
        n_jobs=1,
        remainder="drop",
        verbose=True,
        verbose_feature_names_out=False
    ),
    verbose=True,
).set_output(transform="pandas")

In [ ]:
features_preparation_pipeline1

In [ ]:
prepared_train_features_df1 = (
    features_preparation_pipeline1.fit_transform(train_df)
)

In [ ]:
#prepared_train_features_df1['SalePrice'].max()
prepared_train_features_df1.info()

Feature Engineering and Scaling

In [ ]:
#print(np.isinf(prepared_train_features_df1.select_dtypes()).sum())

In [ ]:


_ = prepared_train_features_df1.hist()
plt.tight_layout()

#This step is done to identify skewed distribution of quantities. 
#The dist. shows x axis(quantity) vs y-axis (no of observations)
#Feature selection is done based on this observation.

In [ ]:
from sklearn import compose

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
from sklearn import linear_model
from sklearn import metrics

In [ ]:
linear_model_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline1,
    compose.TransformedTargetRegressor(
        regressor=linear_model.SGDRegressor(),
        func=safe_log1p,
        inverse_func=safe_expm1
    )
)
## Change1 in pipeline name to roll back

In [ ]:
linear_model_pipeline

In [ ]:
_ = linear_model_pipeline.fit(train_df, train_targets)

In [ ]:
_ = linear_model_pipeline.fit(train_df, train_targets)

train_predictions = linear_model_pipeline.predict(train_df)
train_predictions1 = np.clip(train_predictions, a_min=0, a_max=755000)
print((train_targets <= 0).sum())

train_rmsle = metrics.mean_squared_log_error(
    train_targets,
    train_predictions1,
    squared=False
)

print(f"RMSLE on the training set: {train_rmsle:.5f}")

In [ ]:
from sklearn import model_selection

In [ ]:
from sklearn import tree

In [ ]:
models = {
    "SGDRegressor": linear_model.SGDRegressor(), 
    "DecisionTreeRegressor": tree.DecisionTreeRegressor(),
  
}

ml_pipelines = {}
for name, estimator in models.items():

    ml_pipeline = pipeline.make_pipeline(
        features_preparation_pipeline1,
        compose.TransformedTargetRegressor(
            estimator,
            func=safe_log1p,
            inverse_func=safe_expm1,
        ),
        verbose=True
    )
    ml_pipelines[name] = ml_pipeline

In [ ]:
ml_pipeline_evaluations = {}
for name, ml_pipeline in ml_pipelines.items():
    _ = ml_pipeline.fit(train_df, train_targets)

    # evaluate the pipeline on the training data
    train_predictions = ml_pipeline.predict(train_df)
    train_predictions = np.clip(train_predictions, a_min=0, a_max=755000)
    train_rmsle = metrics.mean_squared_log_error(
        train_targets,
        train_predictions,
        squared=False
    )

    # evaluate the pipeline using cross validation
    train_predictions = model_selection.cross_val_predict(
        ml_pipeline,
        train_df,
        train_targets,
        cv=5,
        n_jobs=-1
    )
    train_predictions = np.clip(train_predictions, a_min=0, a_max=755000)

    cv_rmsle = metrics.mean_squared_log_error(
        train_targets,
        train_predictions,
        squared=False
    )

    ml_pipeline_evaluations[name] = {
        "train_rmsle": train_rmsle,
        "cv_rmsle": cv_rmsle
    }

In [ ]:
ml_pipeline_evaluations_df = (
    pd.DataFrame.from_dict(
        ml_pipeline_evaluations,
        orient="index"
    ).sort_values(
        by="cv_rmsle"
    )
)

In [ ]:
ml_pipeline_evaluations_df

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, sharey=True)

for i, (model, ml_pipeline) in enumerate(ml_pipelines.items()):

    # use CV to generate the predictions
    train_predictions = model_selection.cross_val_predict(
        ml_pipeline,
        train_df,
        train_targets,
        cv=5,
        n_jobs=-1
    )
    prediction_error = train_targets - train_predictions

    # plot the prediction errors against the train_targets
    _ = (
        train_targets.to_frame()
                     .assign(PredictedSalePrice=train_predictions)
                     .assign(PredictionError=prediction_error)
                     .sort_values(by="SalePrice")
                     .plot(
                         kind="scatter",
                         x="SalePrice",
                         y="PredictionError",
                         title=model,
                         alpha=0.1,
                         loglog=True,
                         ax=axes[i],
                         grid=True,
                     )
    )

fig.tight_layout()

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:

# 🔧 RANDOMIZED SEARCH CV for linearmodel (RMSLE Optimization)

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer
from scipy.stats import loguniform, uniform
import numpy as np


def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)  # prevent log of negative predictions
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

rmsle_scorer = make_scorer(rmsle, greater_is_better=False)


y_train_log = np.log1p(train_targets)


# Define pipeline + model

sgd = SGDRegressor(
    random_state=42,
    max_iter=2000,
    tol=1e-3,
)

full_pipeline = make_pipeline(
    features_preparation_pipeline1,  # your preprocessing pipeline
    sgd
)


#  Define search space

param_distributions = {
    "sgdregressor__loss": ["squared_error", "huber", "epsilon_insensitive"],
    "sgdregressor__penalty": ["l2", "l1", "elasticnet"],
    "sgdregressor__alpha": loguniform(1e-6, 1e-2),   # regularization
    "sgdregressor__l1_ratio": uniform(0, 1),         # only used if elasticnet
    "sgdregressor__eta0": loguniform(1e-4, 0.1),     # initial learning rate
    "sgdregressor__learning_rate": ["invscaling", "adaptive"],
}

#
# Randomized Search
# 
search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=40,                 # number of random parameter sets to test
    scoring=rmsle_scorer,
    cv=5,
    n_jobs=-1,
    verbose=False,
    random_state=42,
)

#Fit the model
search.fit(train_df, y_train_log)

#Best Results
print("Best parameters:", search.best_params_)
print("Best RMSLE (CV):", -search.best_score_)

#Training data Predictions
y_pred_log = search.best_estimator_.predict(train_df)
y_pred = np.expm1(y_pred_log)



In [ ]:
#Grid Search CV for Decision Tree

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, mean_squared_log_error
import numpy as np

# --- Custom RMSLE scorer ---
def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, np.maximum(y_pred, 0)))

rmsle_scorer = make_scorer(rmsle, greater_is_better=False)

# --- Base model ---
dt_regressor = DecisionTreeRegressor(random_state=42)

# --- Parameter grid for Decision Tree ---
dt_param_grid = {
    "decisiontreeregressor__max_depth": [3, 5, 7, 10, 15, None],
    "decisiontreeregressor__min_samples_split": [2, 5, 10, 20],
    "decisiontreeregressor__min_samples_leaf": [1, 2, 4, 8],
    "decisiontreeregressor__max_features": [None, "sqrt", "log2"]
}

# --- Pipeline: preprocessing + model ---
dt_pipeline = make_pipeline(features_preparation_pipeline1, dt_regressor)

# --- Grid Search CV setup ---
dt_grid_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    scoring=rmsle_scorer,
    cv=5,
    n_jobs=-1,
    verbose=1
)

# --- Fit the Grid Search ---
dt_grid_search.fit(train_df, y_train_log)

# --- Results ---
print("Best parameters for Decision Tree:")
print(dt_grid_search.best_params_)

print("\nBest RMSLE score (Decision Tree):")
print(-dt_grid_search.best_score_)  # convert from negative to positive RMSLE

# --- Extract best model ---
best_dt_pipeline = dt_grid_search.best_estimator_

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

In [ ]:
""""
1.The model predicts house prices using historical data based on 32 numerical variables out of 79 original, including house area, lot frontage, number of rooms,
and other property characteristics. It provides reasonably accurate predictions and offers insights into which features most strongly influence house prices.

2.Two models were implemented: a linear SGD regressor and a decision tree regressor. The pipeline applies appropriate transformations to handle skewed data. 
The baseline linear model achieved a cross-validated RMSLE of 8.8, which improved significantly to 0.012 after hyperparameter tuning with RandomizedSearchCV and
regularization, demonstrating the substantial impact of regularization.

For the decision tree, a default model yielded a cross-validated RMSLE of 0.23 and a training RMSLE of 0.001, consistent with the overfitting behavior 
typical of decision trees. Hyperparameter tuning with GridSearchCV further improved the model’s cross-validated performance 0.015.


"""

# 9. Submit your predictions to the competition

In [ ]:
y_test_log = search.best_estimator_.predict(test_df)
test_predictions = np.expm1(y_test_log)

In [ ]:

submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice=test_predictions
    ).to_csv(
        "submission.csv",
        index=True,
    )
)


In [ ]:
%%bash

cat submission.csv | head -n 5